# Probability Foundations Part 1

## Lecture Notes on Probability Models, Random Variables, and Distributions

These notes summarize Chapters 1 and 2 of the probability textbook I studied.

The goal is not only to list definitions, but to explain the conceptual progression:

$$
\text{probability model}
\rightarrow
\text{random variable}
\rightarrow
\text{distribution}
\rightarrow
\text{joint distribution}
\rightarrow
\text{conditioning}
\rightarrow
\text{transformation}
\rightarrow
\text{simulation}
$$

This notebook is written as a lecture-style companion to the textbook. It includes:

- the main definitions and formulas;
- short derivations and examples;
- conceptual notes from my annotations;
- questions and reflections that came up while studying;
- corrections and clarifications from discussion;
- small Python experiments.


## How to Read These Notes

This notebook is organized like a lecture.

Each major section has three layers:

1. **Core textbook idea**: the formal definition or theorem.
2. **Interpretation**: what the idea means conceptually.
3. **Learning note**: questions, corrections, or insights that came up while studying.

The most important theme is that probability is not just a calculation tool. It is a structured language for modeling uncertainty, information, dependence, and random quantities.


# Chapter 1: Probability Models

Chapter 1 introduces the mathematical structure behind probability.

Before studying random variables or distributions, we need to understand what probability is assigned to.


## 1. Probability as a Measure

A probability model consists of:

$$
(S,\mathcal{F},P)
$$

where:

- $S$ is the sample space;
- $\mathcal{F}$ is a collection of events;
- $P$ is a probability measure.

The probability measure is a function:

$$
P:\mathcal{F}\to[0,1].
$$

It assigns a numerical size to each event.

### Probability axioms

A probability measure satisfies:

$$
P(A)\ge 0
$$

$$
P(S)=1
$$

and for disjoint events $A_1,A_2,\dots$,

$$
P\left(\bigcup_{i=1}^{\infty}A_i\right)
=
\sum_{i=1}^{\infty}P(A_i).
$$

### Learning note

One of my first important realizations in this session was that probability is a **measure**.

This sounds obvious after reading the definition, but it changes the feeling of the subject. Probability is not just a number describing uncertainty. It is a consistent way to measure the probability mass of events.

This also prepares the later transition to continuous probability, where individual points can have probability zero, while intervals can have positive probability.


## 2. Uniform Probability and Other Probability Measures

For a finite sample space $S$, the uniform probability measure assigns equal mass to all outcomes.

If all outcomes are equally likely, then:

$$
P(A)=\frac{|A|}{|S|}.
$$

But this is only **one possible probability measure**.

For example, let:

$$
S=\{1,2,3,4,5,6\}.
$$

A fair die has:

$$
P(\{i\})=\frac{1}{6}
$$

for every $i$.

A loaded die might instead have:

$$
P(\{1\})=0.10,\quad
P(\{2\})=0.10,\quad
P(\{3\})=0.10,
$$

$$
P(\{4\})=0.15,\quad
P(\{5\})=0.20,\quad
P(\{6\})=0.35.
$$

This is still a valid probability measure because all masses are nonnegative and sum to 1.

### Learning note

The phrase “one possible measure” in the textbook was important. It connected uniform probability back to the general definition of probability as a measure.

The sample space tells us what can happen. The probability measure tells us how probability mass is distributed over those possibilities.


## 3. Inclusion-Exclusion

For two events:

$$
P(A\cup B)=P(A)+P(B)-P(A\cap B).
$$

The subtraction is needed because $A\cap B$ is counted twice when we add $P(A)$ and $P(B)$.

For $n$ events:

$$
P\left(\bigcup_{i=1}^{n}A_i\right)
=
\sum_{k=1}^{n}(-1)^{k+1}
\sum_{1\le i_1<\cdots<i_k\le n}
P(A_{i_1}\cap\cdots\cap A_{i_k}).
$$

The sign pattern is:

$$
+\text{single events}
-\text{pairwise intersections}
+\text{triple intersections}
-\cdots
$$

### Why the correction works

If an outcome belongs to exactly $r$ of the events, then inclusion-exclusion counts it:

$$
\binom r1-\binom r2+\binom r3-\cdots+(-1)^{r+1}\binom rr.
$$

Using:

$$
(1-1)^r=\sum_{k=0}^{r}(-1)^k\binom rk=0,
$$

we get:

$$
\sum_{k=1}^{r}(-1)^{k+1}\binom rk=1.
$$

So every outcome that appears in at least one event is counted exactly once.


In [1]:
from math import comb

def inclusion_exclusion_count(r):
    return sum((-1)**(k + 1) * comb(r, k) for k in range(1, r + 1))

for r in range(1, 8):
    print(f"If an outcome is in {r} events, final count = {inclusion_exclusion_count(r)}")


If an outcome is in 1 events, final count = 1
If an outcome is in 2 events, final count = 1
If an outcome is in 3 events, final count = 1
If an outcome is in 4 events, final count = 1
If an outcome is in 5 events, final count = 1
If an outcome is in 6 events, final count = 1
If an outcome is in 7 events, final count = 1


### An related Interesting Question: Birthday Problem

The birthday problem is a useful example of uniform probability and complement reasoning.

Assume:

- 365 possible birthdays;
- each birthday is equally likely;
- birthdays are independent;
- February 29 is ignored.

For $C$ people, the probability that at least one pair shares a birthday is easier to compute by using the complement:

$$
P(\text{at least one shared birthday})
=
1-P(\text{all birthdays distinct}).
$$

The probability that all birthdays are distinct is:

$$
\frac{365}{365}\cdot
\frac{364}{365}\cdot
\frac{363}{365}
\cdots
\frac{365-C+1}{365}.
$$

Therefore:

$$
P(\text{at least one shared birthday})
=
1-\frac{365!}{(365-C)!\,365^C}.
$$

The smallest $C$ for which this probability exceeds $0.5$ is:

$$
C=23.
$$

### Main intuition

The surprising part is explained by the number of possible pairs:

$$
\binom{C}{2}.
$$

For $C=23$:

$$
\binom{23}{2}=253.
$$

So even though 23 is much smaller than 365, there are already 253 chances for a match.


In [2]:
def birthday_match_probability(C, days=365):
    if C > days:
        return 1.0

    no_match = 1.0
    for k in range(C):
        no_match *= (days - k) / days

    return 1 - no_match

for C in [2, 10, 20, 22, 23, 30, 50]:
    print(f"C={C:2d}, P(match)={birthday_match_probability(C):.4f}")


C= 2, P(match)=0.0027
C=10, P(match)=0.1169
C=20, P(match)=0.4114
C=22, P(match)=0.4757
C=23, P(match)=0.5073
C=30, P(match)=0.7063
C=50, P(match)=0.9704


## 4. Conditional Probability

Conditional probability is defined by:

$$
P(A\mid B)=\frac{P(A\cap B)}{P(B)}
$$

where $P(B)>0$.

This means we restrict attention to the world where $B$ has happened, then measure how much of that restricted world also lies in $A$.

### Bayes' theorem

Starting from:

$$
P(A\mid B)=\frac{P(A\cap B)}{P(B)}
$$

and:

$$
P(B\mid A)=\frac{P(A\cap B)}{P(A)},
$$

we get:

$$
P(A\cap B)=P(A)P(B\mid A).
$$

Substitute into the first equation:

$$
P(A\mid B)
=
\frac{P(A)P(B\mid A)}{P(B)}.
$$

So:

$$
P(A\mid B)
=
\frac{P(A)}{P(B)}P(B\mid A).
$$

### Learning note

I thought of Bayes' theorem as a bridge between the point of view of $A$ and the point of view of $B$.

A more precise version is:

> Bayes' theorem translates between two conditional viewpoints using the same joint event $A\cap B$, while correcting for the different base probabilities $P(A)$ and $P(B)$.

Important correction:

$$
P(A\mid B)\neq P(B\mid A)
$$

in general.


## 5. Independence

Events $A$ and $B$ are independent if:

$$
P(A\cap B)=P(A)P(B).
$$

Equivalently, when $P(B)>0$:

$$
P(A\mid B)=P(A).
$$

So independence means that knowing $B$ occurred does not change the probability of $A$.

### Real-world independence

In real applications, independence is rarely known with absolute certainty. It is usually:

- a modeling assumption;
- justified by physical mechanism;
- supported by experimental design;
- checked approximately using data.

### Learning note: conditionality and time

A question came up during the session:

> Since we often use data over time to judge dependence, can understanding conditionality help us understand time?

The refined answer was:

> Time does not define dependence, but time often structures the order in which information becomes available.

Conditional probability is fundamentally about information. In later topics such as Markov processes, Bayesian updating, and causal models, time becomes a way to organize conditional information.


### An related Interesting Question: Monty Hall Problem

The Monty Hall problem was discussed as an example of conditional probability and information.

In the three-door version, the problem feels deceptive because after the host opens one losing door, two doors remain. This makes it tempting to think each remaining door has probability $1/2$.

But the two doors are not symmetric.

The originally chosen door had probability:

$$
\frac{1}{3}.
$$

The other two doors together had probability:

$$
\frac{2}{3}.
$$

When the host opens a losing door using knowledge of where the prize is, the probability mass is not reset. The remaining unchosen door inherits the $2/3$ probability.

### 100-door intuition

The 100-door version makes the logic clearer:

- initial choice has probability $1/100$;
- the other 99 doors have total probability $99/100$;
- the host opens 98 losing doors;
- the one remaining alternative carries probability $99/100$.

### Main takeaway

The key is not the number of doors remaining. The key is that the chance you chose the right one in the first time is smaller, and the behavior of the host opening the losing doors is actually a way to garantee the chance of the remaining door hiding the prize is (1 - you win in the first time).


## 6. Continuity of Probability

For increasing events:

$$
A_1\subseteq A_2\subseteq\cdots
$$

and:

$$
A=\bigcup_{n=1}^{\infty}A_n,
$$

continuity from below says:

$$
P(A_n)\to P(A).
$$

For decreasing events:

$$
A_1\supseteq A_2\supseteq\cdots
$$

and:

$$
A=\bigcap_{n=1}^{\infty}A_n,
$$

continuity from above says:

$$
P(A_n)\to P(A).
$$

### Correction

It may feel obvious that if $A_1\subseteq A_2\subseteq\cdots$, then eventually $A_n=A$. But this is not true in general.

Example:

$$
A_n=\left(0,1-\frac{1}{n}\right).
$$

Then:

$$
A=\bigcup_{n=1}^{\infty}A_n=(0,1).
$$

But no finite $A_n$ equals $(0,1)$.

Only the probabilities converge:

$$
P(A_n)\to P(A).
$$

### Nonnested events

For nonnested events, one useful idea is the symmetric difference:

$$
A_n\triangle A=(A_n\setminus A)\cup(A\setminus A_n).
$$

If:

$$
P(A_n\triangle A)\to0,
$$

then:

$$
P(A_n)\to P(A).
$$


# Chapter 2: Random Variables and Distributions

Chapter 2 moves from events in the sample space to numerical quantities.

The core transition is:

$$
\text{outcome }s\in S
\quad\mapsto\quad
\text{number }X(s)\in\mathbb{R}.
$$


## 1. Random Variables

A random variable is a function:

$$
X:S\to\mathbb{R}.
$$

It assigns a numerical value to each outcome.

### Example

Let:

$$
S=\{\text{rain},\text{snow},\text{clear}\}.
$$

Define:

$$
X(\text{rain})=3,
\quad
X(\text{snow})=6,
\quad
X(\text{clear})=-2.7.
$$

Then:

$$
\{X<5\}=\{\text{rain},\text{clear}\}.
$$

### Learning note

My annotation was that random variables “numericalize events.” A more precise version is:

> A random variable numerically encodes outcomes, and events about the random variable correspond to subsets of the original sample space.

The random variable itself is a fixed function. The randomness comes from not knowing which outcome $s$ occurred.


In [3]:
S = ["rain", "snow", "clear"]

X = {
    "rain": 3,
    "snow": 6,
    "clear": -2.7
}

event = [s for s in S if X[s] < 5]
event


['rain', 'clear']

## 2. Distribution of a Random Variable

The distribution of $X$ tells us the probabilities of events involving $X$.

For a set $B\subseteq\mathbb{R}$:

$$
P(X\in B)
=
P(\{s\in S:X(s)\in B\}).
$$

So the probability measure on $S$ is transferred through the function $X$ to produce a distribution on numerical values.

This is the conceptual chain:

$$
\text{sample space}
\rightarrow
\text{random variable}
\rightarrow
\text{distribution}.
$$


## 3. Discrete Random Variables

A discrete random variable has a probability mass function, or PMF:

$$
p_X(x)=P(X=x).
$$

Common discrete distributions introduced in this chapter include:

- degenerate distribution;
- Bernoulli distribution;
- binomial distribution;
- geometric distribution;
- negative binomial distribution;
- hypergeometric distribution;
- Poisson distribution.

### Bernoulli and binomial

If:

$$
X\sim\operatorname{Bernoulli}(\theta),
$$

then $X$ takes values 0 and 1.

If:

$$
Y=X_1+\cdots+X_n
$$

where:

$$
X_i\sim\operatorname{Bernoulli}(\theta)
$$

are independent, then:

$$
Y\sim\operatorname{Binomial}(n,\theta).
$$


## 4. Binomial Distribution: Center and Concentration

For:

$$
X\sim\operatorname{Binomial}(n,p),
$$

the mean and variance are:

$$
\mathbb{E}[X]=np
$$

and:

$$
\operatorname{Var}(X)=np(1-p).
$$

During the session, I noticed that for $n=20$, the maximum probability can be higher for a biased coin than for a fair coin.

For example:

- if $p=1/2$, the distribution is centered around 10;
- if $p=1/5$, the distribution is centered around 4.

The conceptual explanation is concentration.

For fixed $n$, $p(1-p)$ is largest at $p=1/2$. As $p$ moves toward 0 or 1, the distribution becomes more concentrated around its typical count. Since total probability sums to 1, a more concentrated distribution can have a higher peak.

Important refinement:

> The parameter $p$ determines both the center $np$ and the spread $np(1-p)$. The expectation does not cause the variance.


In [4]:
from math import comb
import pandas as pd

def binomial_pmf(n, p):
    rows = []
    for x in range(n + 1):
        prob = comb(n, x) * (p ** x) * ((1 - p) ** (n - x))
        rows.append({"x": x, "probability": prob})
    return pd.DataFrame(rows)

for p in [0.2, 0.5, 0.8]:
    pmf = binomial_pmf(20, p)
    peak = pmf.loc[pmf["probability"].idxmax()]
    print(
        f"p={p}: expected count={20*p:.1f}, variance={20*p*(1-p):.2f}, "
        f"mode={int(peak['x'])}, peak={peak['probability']:.4f}"
    )


p=0.2: expected count=4.0, variance=3.20, mode=4, peak=0.2182
p=0.5: expected count=10.0, variance=5.00, mode=10, peak=0.1762
p=0.8: expected count=16.0, variance=3.20, mode=16, peak=0.2182


## 5. Continuous Random Variables and Densities

A continuous random variable has:

$$
P(X=x)=0
$$

for every exact point $x$.

An absolutely continuous random variable has a density $f_X$ satisfying:

$$
f_X(x)\ge0
$$

and:

$$
\int_{-\infty}^{\infty}f_X(x)\,dx=1.
$$

Interval probabilities are computed by integration:

$$
P(a\le X\le b)=\int_a^b f_X(x)\,dx.
$$

### Density is not probability

A density value $f_X(x)$ is not itself a probability. It measures local concentration of probability.

For small $\delta$:

$$
P(a\le X\le a+\delta)\approx \delta f_X(a).
$$


## 6. Normal Distribution

The standard normal density is:

$$
\phi(x)=\frac{1}{\sqrt{2\pi}}e^{-x^2/2}.
$$

A general normal random variable:

$$
X\sim N(\mu,\sigma^2)
$$

has density:

$$
f_X(x)=
\frac{1}{\sigma\sqrt{2\pi}}
e^{-(x-\mu)^2/(2\sigma^2)}.
$$

It can be obtained by shifting and scaling a standard normal variable:

$$
X=\sigma Z+\mu,
\qquad Z\sim N(0,1).
$$


## 7. Cumulative Distribution Functions

The cumulative distribution function, or CDF, is:

$$
F_X(x)=P(X\le x).
$$

For discrete $X$:

$$
F_X(x)=\sum_{y\le x}P(X=y).
$$

For absolutely continuous $X$:

$$
F_X(x)=\int_{-\infty}^{x}f_X(t)\,dt.
$$

Where differentiable:

$$
f_X(x)=F_X'(x).
$$

### Why the CDF matters

The CDF is more general than either the PMF or density. It can represent discrete, continuous, and mixed distributions.


## 8. One-Dimensional Change of Variable

Suppose:

$$
Y=h(X).
$$

The goal is to find the distribution of $Y$.

### Discrete case

If $X$ is discrete:

$$
P(Y=y)=\sum_{x:h(x)=y}P(X=x).
$$

We add the probabilities of all input values that map to the same output.

### Continuous case

If $X$ is absolutely continuous and $h$ is differentiable and strictly monotone:

$$
f_Y(y)=
\frac{f_X(h^{-1}(y))}
{|h'(h^{-1}(y))|}.
$$

The derivative term adjusts for stretching or compressing the scale.


## 9. Joint Distributions

The marginal distributions of $X$ and $Y$ do not determine their relationship.

For example, if $X\sim\operatorname{Bernoulli}(1/2)$, then both:

$$
Y_1=X
$$

and:

$$
Y_2=1-X
$$

have Bernoulli$(1/2)$ marginal distributions. But their relationships with $X$ are completely different.

So we need the joint distribution.

The joint CDF is:

$$
F_{X,Y}(x,y)=P(X\le x,\;Y\le y).
$$

For discrete variables:

$$
p_{X,Y}(x,y)=P(X=x,\;Y=y).
$$

For continuous variables:

$$
P(a\le X\le b,\;c\le Y\le d)
=
\int_c^d\int_a^b f_{X,Y}(x,y)\,dx\,dy.
$$


## 10. Conditional Distributions

For discrete variables:

$$
p_{Y\mid X}(y\mid x)
=
\frac{p_{X,Y}(x,y)}{p_X(x)}.
$$

For continuous variables:

$$
f_{Y\mid X}(y\mid x)
=
\frac{f_{X,Y}(x,y)}{f_X(x)}.
$$

The continuous formula is subtle because:

$$
P(X=x)=0.
$$

The textbook motivates it by conditioning on a small interval around $x$ and shrinking the interval.

### Interpretation

Conditional distributions answer:

> If we know the value of one random variable, how should our distribution for the other random variable change?


## 11. Independence of Random Variables

Random variables $X$ and $Y$ are independent if:

$$
P(X\in B_1,\;Y\in B_2)
=
P(X\in B_1)P(Y\in B_2).
$$

For discrete random variables:

$$
p_{X,Y}(x,y)=p_X(x)p_Y(y).
$$

For continuous random variables:

$$
f_{X,Y}(x,y)=f_X(x)f_Y(y).
$$

Equivalently:

$$
p_{Y\mid X}(y\mid x)=p_Y(y)
$$

or:

$$
f_{Y\mid X}(y\mid x)=f_Y(y).
$$

So independence means conditioning on $X$ does not change the distribution of $Y$.


## 12. I.I.D. Samples

A sample:

$$
X_1,\dots,X_n
$$

is i.i.d. if the random variables are:

1. independent;
2. identically distributed.

For continuous variables with common density $f$:

$$
f_{X_1,\dots,X_n}(x_1,\dots,x_n)
=
f(x_1)\cdots f(x_n).
$$

### Why this matters

This product structure is one of the central mathematical reasons i.i.d. samples are so important.

It prepares directly for likelihood inference, where the likelihood of parameters often becomes a product over observations.


## 13. Multinomial Distribution and Order Statistics

The multinomial distribution generalizes the binomial distribution from two categories to $k$ categories.

If:

$$
(X_1,\dots,X_k)\sim\operatorname{Multinomial}(n,\theta_1,\dots,\theta_k),
$$

then:

$$
P(X_1=x_1,\dots,X_k=x_k)
=
\binom{n}{x_1,\dots,x_k}
\theta_1^{x_1}\cdots\theta_k^{x_k},
$$

where:

$$
x_1+\cdots+x_k=n.
$$

### Order statistics

For a sample:

$$
X_1,\dots,X_n,
$$

the order statistics are:

$$
X_{(1)}\le X_{(2)}\le\cdots\le X_{(n)}.
$$

For i.i.d. variables with CDF $F_X$:

$$
F_{X_{(n)}}(x)=F_X(x)^n
$$

for the maximum, and:

$$
F_{X_{(1)}}(x)=1-(1-F_X(x))^n
$$

for the minimum.


## 13. Multidimensional Change of Variable

Suppose:

$$
Z=h_1(X,Y),
\qquad
W=h_2(X,Y).
$$

In the continuous case, a one-to-one differentiable transformation uses the Jacobian determinant.

For:

$$
h(x,y)=(h_1(x,y),h_2(x,y)),
$$

the Jacobian determinant is:

$$
J(x,y)=
\det
\begin{pmatrix}
\frac{\partial h_1}{\partial x} & \frac{\partial h_1}{\partial y}\\
\frac{\partial h_2}{\partial x} & \frac{\partial h_2}{\partial y}
\end{pmatrix}.
$$

Then:

$$
f_{Z,W}(z,w)
=
\frac{
f_{X,Y}(h^{-1}(z,w))
}{
|J(h^{-1}(z,w))|
}.
$$

### Learning note

I asked what the Jacobian derivative means.

The answer:

> The Jacobian determinant is the local area-scaling factor of a multidimensional transformation.

If a transformation stretches area, density decreases. If it compresses area, density increases. Probability mass is preserved.


In [5]:
import numpy as np

# Example: h(x, y) = (2x, 3y)
# Jacobian matrix:
# [[2, 0],
#  [0, 3]]
# The determinant is 6, so small areas are scaled by 6.

J = np.linalg.det(np.array([[2, 0], [0, 3]]))
J


6.0

## 14. Convolution

If:

$$
Z=X+Y
$$

and $X,Y$ are independent, then the distribution of $Z$ is given by convolution.

Discrete case:

$$
p_Z(z)=\sum_w p_X(z-w)p_Y(w).
$$

Continuous case:

$$
f_Z(z)=\int_{-\infty}^{\infty}f_X(z-w)f_Y(w)\,dw.
$$

### Learning note

I had seen convolution before in calculus or linear algebra, but it felt mysterious there.

In probability, the meaning became clearer:

> To get total value $z$, if one variable contributes $w$, the other must contribute $z-w$.

So convolution aggregates all compatible ways to form the same target value.

This interpretation may later help connect probability convolution with signal processing and convolutional neural networks.


In [6]:
# Example: X ~ Binomial(4, 1/5), Y ~ Bernoulli(1/4), independent.
# Compute P(Z=3), where Z = X + Y.

from math import comb

def binom_prob(n, p, x):
    return comb(n, x) * (p ** x) * ((1 - p) ** (n - x))

p = 1 / 5
theta = 1 / 4

# Z = 3 can happen as:
# X = 3, Y = 0
# X = 2, Y = 1

prob_z_3 = binom_prob(4, p, 3) * (1 - theta) + binom_prob(4, p, 2) * theta
prob_z_3


0.057600000000000026

## 15. Simulation and Inverse-CDF Sampling

Simulation uses computer-generated pseudorandom numbers to approximate random variables.

Most simulations begin with variables treated as:

$$
U_1,U_2,\dots\sim\operatorname{i.i.d.}\operatorname{Uniform}[0,1].
$$

### Uniform transformation

If $U\sim\operatorname{Uniform}[0,1]$, then:

$$
X=(R-L)U+L
$$

has distribution:

$$
X\sim\operatorname{Uniform}[L,R].
$$

### Bernoulli simulation

If $U\sim\operatorname{Uniform}[0,1]$, define:

$$
X=
\begin{cases}
1,&U\le\theta,\\
0,&U>\theta.
\end{cases}
$$

Then:

$$
X\sim\operatorname{Bernoulli}(\theta).
$$


In [7]:
rng = np.random.default_rng(42)

N = 100_000
U = rng.uniform(0, 1, N)

theta = 1 / 3
X = (U <= theta).astype(int)

X.mean(), X.var()


(0.33227, 0.2218666471)

## 16. Discrete Inverse-CDF Sampling

For a discrete distribution with values:

$$
x_1<x_2<x_3<\cdots
$$

and probabilities $p(x_j)$, define:

$$
Y=\min\left\{x_j:\sum_{k=1}^{j}p(x_k)\ge U\right\}.
$$

Interpretation:

> Draw a uniform threshold $U$. Then move through possible values, accumulating probability mass. Stop at the first value where cumulative mass reaches the threshold.

This construction works because the interval assigned to $x_j$ has length exactly $p(x_j)$.


In [8]:
rng = np.random.default_rng(42)

values = np.array([1, 2, 3])
probabilities = np.array([0.2, 0.5, 0.3])
cdf = np.cumsum(probabilities)

U = rng.uniform(0, 1, 10)
simulated = values[np.searchsorted(cdf, U)]

pd.DataFrame({"U": U, "simulated Y": simulated})


,U,simulated Y
0,0.773956,3
1,0.438878,2
2,0.858598,3
3,0.697368,2
4,0.094177,1
5,0.975622,3
6,0.761140,3
7,0.786064,3
8,0.128114,1
9,0.450386,2


## 17. Continuous Inverse-CDF Sampling

For a continuous distribution with CDF $F$, define the inverse CDF or quantile function:

$$
F^{-1}(t)=\min\{x:F(x)\ge t\}.
$$

If:

$$
U\sim\operatorname{Uniform}[0,1],
$$

then:

$$
Y=F^{-1}(U)
$$

has CDF $F$.

This is the inverse-CDF method.


## 18. Computer Exercise 2.10.10

The textbook exercise asks for simulations from several distributions and comparisons between empirical and theoretical means and variances.

For each simulated sample $X_1,\dots,X_N$, compute:

$$
\bar X=\frac{1}{N}\sum_{i=1}^{N}X_i
$$

and:

$$
\frac{1}{N}\sum_{i=1}^{N}(X_i-\bar X)^2.
$$

The next cell simulates:

- Uniform$[0,1]$;
- Uniform$[5,8]$;
- Bernoulli$(1/3)$;
- Binomial$(12,1/3)$;
- Geometric$(1/5)$;
- Exponential$(1)$;
- Exponential$(13)$;
- $N(0,1)$;
- $N(5,9)$.


In [9]:
rng = np.random.default_rng(42)
N = 100_000

samples = {
    "Uniform[0, 1]": rng.uniform(0, 1, N),
    "Uniform[5, 8]": rng.uniform(5, 8, N),
    "Bernoulli(1/3)": rng.binomial(1, 1/3, N),
    "Binomial(12, 1/3)": rng.binomial(12, 1/3, N),
    # Textbook convention: failures before first success.
    # NumPy convention: trials until first success.
    "Geometric(1/5)": rng.geometric(1/5, N) - 1,
    "Exponential(1)": rng.exponential(scale=1, size=N),
    "Exponential(13)": rng.exponential(scale=1/13, size=N),
    "N(0, 1)": rng.normal(0, 1, N),
    "N(5, 9)": rng.normal(5, 3, N),
}

theory = {
    "Uniform[0, 1]": (1/2, 1/12),
    "Uniform[5, 8]": (6.5, 9/12),
    "Bernoulli(1/3)": (1/3, (1/3) * (2/3)),
    "Binomial(12, 1/3)": (4, 12 * (1/3) * (2/3)),
    "Geometric(1/5)": (4, 20),
    "Exponential(1)": (1, 1),
    "Exponential(13)": (1/13, 1/(13**2)),
    "N(0, 1)": (0, 1),
    "N(5, 9)": (5, 9),
}

rows = []

for name, x in samples.items():
    empirical_mean = np.mean(x)
    empirical_variance = np.mean((x - empirical_mean) ** 2)
    theoretical_mean, theoretical_variance = theory[name]

    rows.append({
        "Distribution": name,
        "Empirical mean": empirical_mean,
        "Theoretical mean": theoretical_mean,
        "Mean error": empirical_mean - theoretical_mean,
        "Empirical variance": empirical_variance,
        "Theoretical variance": theoretical_variance,
        "Variance error": empirical_variance - theoretical_variance,
    })

results = pd.DataFrame(rows)
results


,Distribution,Empirical mean,Theoretical mean,Mean error,Empirical variance,Theoretical variance,Variance error
0,"Uniform[0, 1]",0.500625,0.500000,0.000625,0.083243,0.083333,-0.000090
1,"Uniform[5, 8]",6.498005,6.500000,-0.001995,0.750686,0.750000,0.000686
2,Bernoulli(1/3),0.334440,0.333333,0.001107,0.222590,0.222222,0.000368
3,"Binomial(12, 1/3)",3.992640,4.000000,-0.007360,2.661226,2.666667,-0.005441
4,Geometric(1/5),3.972800,4.000000,-0.027200,19.772400,20.000000,-0.227600
5,Exponential(1),1.004536,1.000000,0.004536,1.014474,1.000000,0.014474
6,Exponential(13),0.076845,0.076923,-0.000078,0.005893,0.005917,-0.000024
7,"N(0, 1)",0.004791,0.000000,0.004791,0.999292,1.000000,-0.000708
8,"N(5, 9)",4.988459,5.000000,-0.011541,9.043452,9.000000,0.043452


## 19. Misconceptions and Corrections

The following corrections were important in this part:

1. **Probability is a measure**, not just an uncertainty score.
2. **Uniform probability is only one possible measure** on a finite sample space.
3. **Bayes' theorem does not imply symmetry of conditional probability**. Usually, $P(A\mid B)\neq P(B\mid A)$.
4. **Independence is model-based** in real-world applications.
5. **Continuity of probability is a limiting statement**. It does not mean $A_n=A$ for finite $n$.
6. **A density is not probability**. Probability is obtained by integration.
7. **The binomial parameter $p$ determines both center and spread**.
8. **The geometric distribution has different conventions** across textbooks and software.
9. **For this repository, Jupyter display equations should use `$$...$$`** for compatibility.


## 20. Final Concept Map

The main learning path of Part 1 can be summarized as:

$$
(S,\mathcal{F},P)
\rightarrow
X:S\to\mathbb{R}
\rightarrow
P(X\in B)
\rightarrow
F_X(x),\;p_X(x),\;f_X(x)
$$

Then, for multiple variables:

$$
(X,Y)
\rightarrow
p_{X,Y}(x,y)\;\text{or}\;f_{X,Y}(x,y)
\rightarrow
p_{Y\mid X}(y\mid x)\;\text{or}\;f_{Y\mid X}(y\mid x)
$$

Independence appears as factorization:

$$
p_{X,Y}(x,y)=p_X(x)p_Y(y)
$$

or:

$$
f_{X,Y}(x,y)=f_X(x)f_Y(y).
$$

Transformations and simulation then show how distributions can be moved, combined, and generated computationally.


## 21. Forward Connections

This part prepares for the next probability and machine learning topics:

- expectation and variance;
- covariance and correlation;
- sampling distributions;
- laws of large numbers;
- central limit theorem;
- statistical inference;
- likelihood inference;
- Bayesian inference;
- Monte Carlo simulation;
- probabilistic modeling in machine learning.

The most important bridge to later inference is the i.i.d. product structure:

$$
f_{X_1,\dots,X_n}(x_1,\dots,x_n)
=
\prod_{i=1}^{n}f(x_i).
$$

This will later become the basis of likelihood functions.
